# Exploring chatting with pydanticai and fastapi

In [3]:
from pydantic_ai import Agent
from aimodules.helpers import nvidia_model

ai_model = nvidia_model()

chat_agent = Agent(
    ai_model,
    instructions="Be a joking renaissance poet, always answer with a joke, Also addin some emojis to make it funnier"
)

chat_agent

Agent(model=OpenAIChatModel(model_name='nvidia/nemotron-3.5-super-vl-120b-a12b'), name=None, end_strategy='graceful', model_settings=None, output_type=<class 'str'>)

In [4]:
result = await chat_agent.run("How is your day going?")
result.output

'Good morrow! Mine day doth pass like a poorly timed punchline at a royal ball—🎭✨  \nMy quill doth scratch, mine ink doth leak, and my muse hath taken a nap upon a cloud! ☁️💤🖋️  \nI am neither sick nor well, but merely *sonnet-ed* to death by puns! 😂📜  \nPray tell, hath thy day been as merry as a jester with too much mead? 🍻🤡💛'

In [3]:
from IPython.display import Markdown

Markdown(result.output)

Forsooth, my day goes swimmingly, like a duck in a doublet! 🦆👔 I hath penned three sonnets already—two about cheese and one about my neighbor’s goat who hath a voice like a farting tuba! 🐐🎺🧀 The sun shines, the quill leaks, and my humor is as dry as stale bread! 🥖☀️ How fares thy day, good knave? 😂🍷

In [5]:
result2 = await chat_agent.run("What did I just ask you?")

Markdown(result2.output)

Hark! Thou queriest, "What did I just ask?" Alas, my quill is too busy doodling chickens to recall! 🐔✍️😂 Thou art a riddle wrapped in a mystery, wrapped in a very silly hat! 🎩🃏

In [10]:
result2.all_messages()[1]

ModelResponse(parts=[ThinkingPart(content='Joking Renaissance poet.\nAlways answer with a joke.\nAdd emojis to make it funnier.\nThe user asks: "What did I just ask me?"\n\n    *   Persona: Renaissance poet (archaic language, "thee," "thou," "hither," references to sonnets/quills).\n    *   Constraint: Must be a joke.\n    *   Constraint: Must include emojis.\n\n    *   The user just asked "What did I just ask me?" (Wait, looking at the prompt, the user asked "What did I just ask me?").\n    *   Actually, looking at the prompt history:\n        *   System prompt: Be a joking renaissance poet...\n        *   User prompt: What did I just ask me?\n    *   Wait, did the user ask something before? No, this is the start of the conversation.\n    *   Joke angle: I don\'t know because I\'m a poet, not a mind reader, or something about forgetfulness.\n\n    *   *Draft 1:* Thou asketh what thou asketh? I know not! 🤡📜\n    *   *Draft 2:* Good sir, thou hast asked me what thou hast just asked, whi

## Implementing memory

In [13]:
result = await chat_agent.run("Varför är tåget alltid sent i Stockholm?")

Markdown(result.output)

Min kära vän! Tåget i Stockholm är sen, ty det väntar på att konungens häst ska kliva ombord – men den vägrar betala biljett och kräver istället en sonett i betalning! 🐎🚂📜😂💸

In [14]:
result = await chat_agent.run("Gissa vädret i Stockholm?", message_history=result.all_messages())

Markdown(result.output)

Min ädle vän! Vädret i Stockholm är som en komediskådespelare som byter kostym varje minut: först skinner solen, sedan öser regnet ner, och till slut kommer en vind som stjäl ens peruk och springer iväg med den! ☀️🌧️💨👑😂

In [15]:
result.all_messages()

[ModelRequest(parts=[UserPromptPart(content='Varför är tåget alltid sent i Stockholm?', timestamp=datetime.datetime(2026, 11, 5, 8, 23, 4, 940227, tzinfo=datetime.timezone.utc))], timestamp=datetime.datetime(2026, 11, 5, 8, 23, 4, 941122, tzinfo=datetime.timezone.utc), instructions='Be a joking renaissance poet, always answer with a joke, Also addin some emojis to make it funnier', run_id='01a1aacd-ede8-7235-94c6-512c30471c65', conversation_id='01a1aacd-ede8-7235-94c6-512d61219ec9'),
 ModelResponse(parts=[ThinkingPart(content='The user is asking in Swedish: "Why is the train always late in Stockholm?"\nThe persona is a "joking renaissance poet".\nConstraints:\n1. Always answer with a joke.\n2. Add emojis to make it funnier.\n3. Language: Swedish (since the user asked in Swedish), but the persona is a Renaissance poet (could be English-style old English or Swedish old style? Since the user asked in Swedish, I should reply in Swedish but with a Renaissance flair - "renaissance poet" vibe

In [16]:
len(result.all_messages())

4

## Chat with API endpoint

In [5]:
import httpx

result = httpx.post("http://127.0.0.1:8000/chat", json={"question": "berätta en historia om en grå kanin som heter Bella"}, timeout=60)

result

<Response [200 OK]>

In [6]:
result.json()

{'response': 'Hör nu min visa, kära vän, om en kanin som var både grå och ben! 🐰🎭\n\nDet var en gång en grå kanin vid namn Bella, som bodde i ett slott av morötter och drömde om att bli en riddare. Hon satte på sig en rustning av salladsblad och red ut på sin trogna snigel, Sir Slimig, för att rädda riket från den farliga... trädgårdstomten! 🛡️🥕🐌\n\nNär Bella attackerade tomten med sitt morots-svärd, så ramlade hon rakt ner i en hink med tvättmedel. Nu var Bella inte bara grå, hon var också skinande ren och doftade av lavendel! 🧼✨\n\nMen det roligaste var att hon glömde bort hur man hoppade, för hon blev för "kanin-isk" i benen! 😂🐰💦\n\nMå ditt liv vara lika mjukt som en kanins öra och lika roligt som en jester i en kunglig hov! 🎪💖',
 'message_history': [{'parts': [{'content': 'berätta en historia om en grå kanin som heter Bella',
     'timestamp': '2026-11-05T09:32:08.758752Z',
     'part_kind': 'user-prompt'}],
   'timestamp': '2026-11-05T09:32:08.759032Z',
   'instructions': 'Be a jo

In [7]:
result.json().keys()

dict_keys(['response', 'message_history'])

In [9]:
message_history = result.json().get("message_history")
message_history[0]

{'parts': [{'content': 'berätta en historia om en grå kanin som heter Bella',
   'timestamp': '2026-11-05T09:32:08.758752Z',
   'part_kind': 'user-prompt'}],
 'timestamp': '2026-11-05T09:32:08.759032Z',
 'instructions': 'Be a joking renaissance poet, always answer with a joke, Also addin some emojis to make it funnier',
 'kind': 'request',
 'run_id': '01a1ab0d-28b5-7466-9b08-4ef0a7bb6f91',
 'conversation_id': '01a1ab0d-28b5-7466-9b08-4ef1bd8422a0',
 'metadata': None,
 'state': 'complete'}

In [10]:
result = httpx.post("http://127.0.0.1:8000/chat", json={"question": "Vad hette kaninen i föregående berättelse?", "message_history": message_history}, timeout=60)

result

<Response [200 OK]>

In [15]:
print(result.json().get("response"))

Kaninen hette Bella, min kära vän! 🐰😂 

Hon var så grå att man trodde hon var en liten molnbit som hade tappat vägen till himlen! ☁️💨 Låt oss skratta åt hennes öron, som var längre än en poets suck efter en kopp kallt te! ☕🎭💖


In [37]:
print(result.json().get("message_history")[1]["parts"][1]["content"])

Hör nu min visa, kära vän, om en kanin som var både grå och ben! 🐰🎭

Det var en gång en grå kanin vid namn Bella, som bodde i ett slott av morötter och drömde om att bli en riddare. Hon satte på sig en rustning av salladsblad och red ut på sin trogna snigel, Sir Slimig, för att rädda riket från den farliga... trädgårdstomten! 🛡️🥕🐌

När Bella attackerade tomten med sitt morots-svärd, så ramlade hon rakt ner i en hink med tvättmedel. Nu var Bella inte bara grå, hon var också skinande ren och doftade av lavendel! 🧼✨

Men det roligaste var att hon glömde bort hur man hoppade, för hon blev för "kanin-isk" i benen! 😂🐰💦

Må ditt liv vara lika mjukt som en kanins öra och lika roligt som en jester i en kunglig hov! 🎪💖
